# Pipeline Klasifikasi Risiko ISPA — Jawa Timur 2024

Notebook ini menyatukan seluruh alur kerja proyek dalam satu tempat: dari **data mentah** (PDF Profil Kesehatan, PDF Dalam Angka, Excel Satu Data Jatim) sampai **dataset analisis** `data/processed/dataset_final_2024.csv`, lalu clustering dan klasifikasi.

| Bagian | Isi | Keluaran |
|---|---|---|
| 0 | Setup path & pustaka | — |
| 1 | Ekstraksi Tabel 61 & 84 Profil Kesehatan 2024 | `pneumonia_2024.csv`, `sanitasi_2024.csv` |
| 2 | Ekstraksi kepadatan penduduk (Dalam Angka 2025, Tabel 3.1.1) | `kepadatan_2024.csv` |
| 3 | IKU dari Excel Satu Data Jatim | `iku_jatim_2024.csv` |
| 4 | Agregasi 4 sumber → 38 kab/kota × 5 variabel | **`dataset_final_2024.csv`** |
| 5 | Clustering: K-Means vs Ward (k = 2..6) | tabel silhouette |
| 6 | Pelabelan rendah/sedang/tinggi + uji stabilitas | label per kab/kota |
| 7 | Klasifikasi pembanding (LOOCV) | akurasi & macro-F1 |
| 8 | Simpan keluaran | `output/cluster_labels_2024.csv` |

**Cara pakai:** buka dari root proyek (`E:\DATA WRANGLING\CLustering ISPA_UAS`) lalu *Run All*. Pustaka yang dibutuhkan: `pandas`, `numpy`, `pymupdf`, `openpyxl`, `scikit-learn`.

**Dua blok variabel** yang dipakai konsisten di seluruh notebook:

- **Blok A — pembentuk label:** `rate_pneu`, `prop_berat_pct`. Keduanya *outcome* (hasil akhir), jadi tidak boleh dipakai sebagai prediktor — itu akan membuat klasifikasi sirkular.
- **Blok B — prediktor:** `iku`, `sanitasi`, `kepadatan`. Karakteristik konteks wilayah yang diukur independen dari outcome.

## Bagian 0 — Setup

Semua path diturunkan dari `Path.cwd()`, jadi notebook harus dijalankan dari root proyek. Kalau ada berkas yang hilang, sel ini akan menandainya sebelum pipeline mulai.

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, adjusted_rand_score,
                             classification_report, f1_score, silhouette_score)
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
REF = ROOT / "data" / "reference"
OUT = ROOT / "output"
for d in (PROC, OUT):
    d.mkdir(parents=True, exist_ok=True)

TAHUN = 2024
DIM = REF / "dim_kabupaten_jatim.csv"
PDF_PROFIL = RAW / "profil_kesehatan_jatim_2024.pdf"
PDF_ANGKA = RAW / "jatim_dalam_angka_2025.pdf"
XLSX_IKU = RAW / "indeks-kualitas-udara-tahun-2024-250813689cc4170e91e.xlsx"

print(f"root proyek: {ROOT}")
for p in (DIM, PDF_PROFIL, PDF_ANGKA, XLSX_IKU):
    print(("  OK      " if p.exists() else "  HILANG  ") + str(p.relative_to(ROOT)))

root proyek: E:\DATA WRANGLING\CLustering ISPA_UAS
  OK      data\reference\dim_kabupaten_jatim.csv
  OK      data\raw\profil_kesehatan_jatim_2024.pdf
  OK      data\raw\jatim_dalam_angka_2025.pdf
  OK      data\raw\indeks-kualitas-udara-tahun-2024-250813689cc4170e91e.xlsx


## Bagian 1 — Ekstraksi Tabel Profil Kesehatan Jawa Timur 2024

Sumber: `data/raw/profil_kesehatan_jatim_2024.pdf`.

Nomor halaman tabel berbeda antar-edisi, jadi tabel dicari lewat **judul kolomnya**, bukan nomor halaman. Tiga fungsi bantu dulu:

- `ke_angka` — mengubah format Indonesia `"1.234,5"` menjadi `1234.5`, dan `"-"` menjadi `NaN`.
- `pecah_wilayah` — memecah `"KAB. PACITAN"` menjadi `("Kabupaten", "Pacitan")` supaya bisa di-join ke tabel dimensi.
- `cari_tabel` — memindai halaman yang memuat semua kata penanda, lalu mengambil tabel pertama dengan jumlah baris memadai.

In [2]:
def ke_angka(x):
    """'1.234,5' -> 1234.5 ; kosong / '-' -> NaN."""
    if x is None:
        return float("nan")
    s = str(x).strip().replace(" ", "")
    if s in ("", "-", "nan", "None"):
        return float("nan")
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return float("nan")


def pecah_wilayah(label):
    """'KAB. PACITAN' -> ('Kabupaten', 'Pacitan') ; 'KOTA BATU' -> ('Kota', 'Batu')."""
    s = str(label).strip()
    m = re.match(r"(?i)^(KAB\.?|KABUPATEN|KOTA)\s+(.*)$", s)
    if not m:
        return None, None
    jenis = "Kota" if m.group(1).upper().startswith("KOTA") else "Kabupaten"
    return jenis, m.group(2).strip().title()


def cari_tabel(doc, penanda, min_baris=30):
    """Halaman + baris tabel pertama yang memuat semua kata penanda."""
    for i, page in enumerate(doc, start=1):
        teks = page.get_text()
        if not all(p in teks for p in penanda):
            continue
        for t in page.find_tables().tables:
            rows = t.extract()
            if len(rows) >= min_baris:
                return i, rows
    return None, None


doc = pymupdf.open(PDF_PROFIL)
print("halaman PDF:", doc.page_count)

halaman PDF: 400


### 1.1 Tabel 61 — pneumonia balita

Dua variabel outcome dihitung dari tabel ini:

$$\text{rate\_pneu} = \frac{\text{total penemuan}}{\text{jumlah balita}} \times 1000
\qquad
\text{prop\_berat\_pct} = \frac{\text{pneumonia berat}}{\text{total penemuan}} \times 100$$

`rate_pneu` = penemuan kasus per 1.000 balita (bukan insiden sesungguhnya, karena pembilangnya adalah kasus yang **dilaporkan**). `prop_berat_pct` = proporsi kasus berat, penanda derajat keparahan.

In [3]:
hal_pneu, rows_pneu = cari_tabel(doc, ["PNEUMONIA", "JUMLAH BALITA"])
assert rows_pneu is not None, "Tabel pneumonia tidak ditemukan"

rec = []
for r in rows_pneu:
    jenis, nama = pecah_wilayah(r[1]) if len(r) > 1 else (None, None)
    if jenis is None:
        continue
    rec.append({
        "jenis": jenis,
        "nama_wilayah": nama,
        "jumlah_balita": ke_angka(r[3]),
        "perkiraan_pneumonia": ke_angka(r[7]),
        "pneumonia_lp": ke_angka(r[12]),
        "pneu_berat_lp": ke_angka(r[13]),
        "total_penemuan": ke_angka(r[14]),
        "cakupan_pct": ke_angka(r[15]),
    })

pneu = pd.DataFrame(rec)
pneu["rate_pneu"] = (pneu["total_penemuan"] / pneu["jumlah_balita"] * 1000).round(2)
pneu["rate_perkiraan"] = (pneu["perkiraan_pneumonia"] / pneu["jumlah_balita"] * 1000).round(2)
pneu["prop_berat_pct"] = (pneu["pneu_berat_lp"] / pneu["total_penemuan"] * 100).round(2)

print(f"tabel di halaman {hal_pneu} | baris wilayah terbaca: {len(pneu)}")
pneu[["nama_wilayah", "jumlah_balita", "total_penemuan", "rate_pneu", "prop_berat_pct"]].head(8)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


tabel di halaman 368 | baris wilayah terbaca: 38


,nama_wilayah,jumlah_balita,total_penemuan,rate_pneu,prop_berat_pct
0,Pacitan,35801.0,446.0,12.46,46.64
1,Ponorogo,62363.0,1766.0,28.32,46.26
2,Trenggalek,48239.0,1537.0,31.86,43.98
3,Tulungagung,74308.0,2259.0,30.40,49.09
4,Blitar,83700.0,2570.0,30.70,49.11
5,Kediri,123618.0,2248.0,18.19,47.82
6,Malang,190205.0,8428.0,44.31,47.64
7,Lumajang,79430.0,1664.0,20.95,48.20


### 1.2 Tabel 84 — sanitasi layak

`sanitasi` = persentase rumah tangga dengan akses sanitasi layak; `babs_terbuka_pct` = persentase buang air besar sembarangan (kolom pembanding, tidak masuk model).

In [4]:
hal_san, rows_san = cari_tabel(doc, ["SANITASI", "JUMLAH KK"])
assert rows_san is not None, "Tabel sanitasi tidak ditemukan"

rec = []
for r in rows_san:
    jenis, nama = pecah_wilayah(r[1]) if len(r) > 1 else (None, None)
    if jenis is None:
        continue
    rec.append({
        "jenis": jenis,
        "nama_wilayah": nama,
        "jumlah_kk": ke_angka(r[4]),
        "sanitasi": ke_angka(r[8]),
        "sanitasi_aman_pct": ke_angka(r[6]),
        "babs_terbuka_pct": ke_angka(r[16]) if len(r) > 16 else float("nan"),
    })

sanitasi = pd.DataFrame(rec)

print(f"tabel di halaman {hal_san} | baris wilayah terbaca: {len(sanitasi)}")
sanitasi[["nama_wilayah", "jumlah_kk", "sanitasi", "babs_terbuka_pct"]].head(8)

tabel di halaman 397 | baris wilayah terbaca: 38


,nama_wilayah,jumlah_kk,sanitasi,babs_terbuka_pct
0,Pacitan,194892.0,81.97,0.0
1,Ponorogo,335313.0,86.18,0.0
2,Trenggalek,241359.0,83.80,0.0
3,Tulungagung,375351.0,82.78,0.0
4,Blitar,459648.0,82.93,0.0
5,Kediri,527153.0,82.39,0.0
6,Malang,853974.0,84.91,0.0
7,Lumajang,323677.0,79.22,0.0


### 1.3 Gabung ke tabel dimensi & simpan

Tabel dimensi (`dim_kabupaten_jatim.csv`) adalah kunci join: 38 kabupaten/kota dengan `kode_bps` 4 digit. Kolom `tahun` di tabel dimensi **dibuang** dan diganti `tahun_sumber` yang dibaca dari dokumen sumber, supaya tidak salah klaim tahun.

In [5]:
dim = pd.read_csv(DIM, dtype={"kode_bps": str})


def gabung_dim(df, label):
    m = dim.merge(df, on=["jenis", "nama_wilayah"], how="left", validate="one_to_one")
    m = m.drop(columns=["tahun"], errors="ignore")
    m.insert(3, "tahun_sumber", TAHUN)
    print(f"{label:10s} | sumber={len(df):2d} | hasil join={len(m):2d} | "
          f"baris tak cocok={m['nama_wilayah'].isna().sum()}")
    assert len(m) == 38, "hasil join harus 38 kabupaten/kota"
    assert m["nama_wilayah"].notna().all(), "ada wilayah yang tidak cocok dengan dimensi"
    return m


pneu_j = gabung_dim(pneu, "pneumonia")
san_j = gabung_dim(sanitasi, "sanitasi")

pneu_j.to_csv(PROC / f"pneumonia_{TAHUN}.csv", index=False)
san_j.to_csv(PROC / f"sanitasi_{TAHUN}.csv", index=False)
print("\ntersimpan:", PROC / f"pneumonia_{TAHUN}.csv")
print("tersimpan:", PROC / f"sanitasi_{TAHUN}.csv")

pneumonia  | sumber=38 | hasil join=38 | baris tak cocok=0
sanitasi   | sumber=38 | hasil join=38 | baris tak cocok=0

tersimpan: E:\DATA WRANGLING\CLustering ISPA_UAS\data\processed\pneumonia_2024.csv
tersimpan: E:\DATA WRANGLING\CLustering ISPA_UAS\data\processed\sanitasi_2024.csv


## Bagian 2 — Kepadatan penduduk (Dalam Angka 2025)

Sumber: `data/raw/jatim_dalam_angka_2025.pdf`, Tabel 3.1.1. Tabel memuat tiga kolom tahun (2020, 2024, 2025); yang diambil kolom **2024**.

Dua jebakan yang ditangani:

1. Halaman sebelumnya juga memuat frasa "Kepadatan Penduduk" di judul, tapi isinya jumlah penduduk (jiwa) — karena itu header tabel ikut diperiksa.
2. Teks watermark ikut terbaca di sel (`"d\ni\n1.109"`), jadi angka dibersihkan dengan regex dan nama diambil baris terakhir sel.

In [6]:
def bersih_angka(x):
    """'d\\ni\\n1.109' -> 1109 (buang sisa watermark, lalu titik ribuan)."""
    if x is None:
        return None
    s = re.sub(r"[^0-9.]", "", str(x))
    if not s or s == ".":
        return None
    return int(s.replace(".", ""))


def bersih_nama(x):
    """'s\\nMagetan' -> 'Magetan'."""
    if x is None:
        return None
    bagian = [b.strip() for b in str(x).split("\n") if b.strip()]
    return bagian[-1].title() if bagian else None


def cari_tabel_kepadatan(doc, penanda):
    for i, page in enumerate(doc, start=1):
        teks = page.get_text()
        if not all(p in teks for p in penanda):
            continue
        for t in page.find_tables().tables:
            rows = t.extract()
            if len(rows) < 35:
                continue
            header = " ".join(str(c) for c in rows[0] if c).lower()
            if "kepadatan" in header:
                return i, rows
    return None, None


doc_angka = pymupdf.open(PDF_ANGKA)
hal_kep, rows_kep = cari_tabel_kepadatan(doc_angka, ["Kepadatan Penduduk", "Kabupaten/Kota"])
assert rows_kep is not None, "Tabel kepadatan tidak ditemukan"

# kolom: 0 nomor, 1 nama, 2 = 2020, 3 = 2024, 4 = 2025
KOLOM_TAHUN = {2020: 1, 2024: 2, 2025: 3}
kolom = KOLOM_TAHUN[TAHUN]

jenis = None
rec = []
for r in rows_kep:
    sel0 = str(r[0]).lower() if r[0] else ""
    if "kabupaten/regency" in sel0:
        jenis = "Kabupaten"
        continue
    if "kota/municipality" in sel0:
        jenis = "Kota"
        continue
    if "jawa timur" in sel0:
        break
    if jenis is None:
        continue
    nilai = bersih_angka(r[kolom])
    nama = bersih_nama(r[0])
    if nama and nilai is not None:
        rec.append({"jenis": jenis, "nama_wilayah": nama, "kepadatan": nilai})

kep = pd.DataFrame(rec)
kep_j = dim[["kode_bps", "nama_wilayah", "jenis"]].merge(
    kep, on=["jenis", "nama_wilayah"], how="left", validate="one_to_one")
kep_j.insert(3, "tahun_sumber", TAHUN)

print(f"tabel di halaman {hal_kep} | baris terbaca={len(kep)} | hasil join={len(kep_j)} | "
      f"kosong={kep_j['kepadatan'].isna().sum()}")
print(f"kepadatan: min={kep_j.kepadatan.min():.0f} | median={kep_j.kepadatan.median():.0f} | "
      f"max={kep_j.kepadatan.max():.0f}")

kep_j.to_csv(PROC / f"kepadatan_{TAHUN}.csv", index=False)
print("tersimpan:", PROC / f"kepadatan_{TAHUN}.csv")
kep_j.head(8)

tabel di halaman 126 | baris terbaca=38 | hasil join=38 | kosong=0
kepadatan: min=411 | median=862 | max=8698
tersimpan: E:\DATA WRANGLING\CLustering ISPA_UAS\data\processed\kepadatan_2024.csv


,kode_bps,nama_wilayah,jenis,tahun_sumber,kepadatan
0,3501,Pacitan,Kabupaten,2024,411
1,3502,Ponorogo,Kabupaten,2024,679
2,3503,Trenggalek,Kabupaten,2024,596
3,3504,Tulungagung,Kabupaten,2024,973
4,3505,Blitar,Kabupaten,2024,724
5,3506,Kediri,Kabupaten,2024,1109
6,3507,Malang,Kabupaten,2024,788
7,3508,Lumajang,Kabupaten,2024,638


## Bagian 3 — Indeks Kualitas Udara (IKU)

Sumber: Excel Satu Data Jawa Timur `indeks-kualitas-udara-tahun-2024-*.xlsx`, satu sheet bernama `data`. Kolomnya sudah rapi, jadi tidak perlu ekstraksi PDF — cukup di-rename lalu di-join ke dimensi.

Satu baris berlebih di berkas sumber adalah **total Provinsi Jawa Timur** (kodenya bukan 4 digit). Baris itu dibuang lebih dulu supaya agregasi hanya berisi 38 kabupaten/kota.

In [7]:
iku_raw = pd.read_excel(XLSX_IKU, sheet_name="data")
iku_raw = iku_raw.rename(columns={
    "kode_kabupaten_kota": "kode_bps",
    "indeks_kualitas_udara": "iku",
})
iku_raw["kode_bps"] = iku_raw["kode_bps"].astype(str)

print("tahun pada sumber:", sorted(iku_raw["tahun"].unique()), "| baris mentah:", len(iku_raw))

# buang baris total provinsi (kode bukan 4 digit)
bukan_kabkota = ~iku_raw["kode_bps"].str.fullmatch(r"\d{4}")
print("baris dibuang:", iku_raw.loc[bukan_kabkota, "nama_kabupaten_kota"].tolist())
iku_raw = iku_raw.loc[~bukan_kabkota]

iku = dim[["kode_bps", "nama_wilayah", "jenis"]].merge(
    iku_raw[["kode_bps", "iku"]], on="kode_bps", how="left", validate="one_to_one")

print(f"hasil join={len(iku)} | kosong={iku['iku'].isna().sum()}")
print(f"IKU: min={iku.iku.min():.2f} | median={iku.iku.median():.2f} | max={iku.iku.max():.2f}")

iku.to_csv(PROC / f"iku_jatim_{TAHUN}.csv", index=False)
print("tersimpan:", PROC / f"iku_jatim_{TAHUN}.csv")
iku.head(8)

tahun pada sumber: [np.int64(2024)] | baris mentah: 39
baris dibuang: ['PROVINSI JAWA TIMUR']
hasil join=38 | kosong=0
IKU: min=71.91 | median=89.00 | max=94.84
tersimpan: E:\DATA WRANGLING\CLustering ISPA_UAS\data\processed\iku_jatim_2024.csv


,kode_bps,nama_wilayah,jenis,iku
0,3501,Pacitan,Kabupaten,92.06
1,3502,Ponorogo,Kabupaten,94.09
2,3503,Trenggalek,Kabupaten,94.84
3,3504,Tulungagung,Kabupaten,91.42
4,3505,Blitar,Kabupaten,88.06
5,3506,Kediri,Kabupaten,94.31
6,3507,Malang,Kabupaten,91.14
7,3508,Lumajang,Kabupaten,92.07


## Bagian 4 — Agregasi menjadi `dataset_final_2024.csv`

Empat tabel digabung dengan `kode_bps` sebagai kunci tunggal, memakai `validate="one_to_one"` supaya penggabungan yang salah langsung gagal, bukan diam-diam menghasilkan baris ganda.

Hasil akhir: **38 baris × 5 variabel** (2 pembentuk label + 3 prediktor).

In [8]:
final = (iku[["kode_bps", "iku"]]
         .merge(pneu_j[["kode_bps", "rate_pneu", "prop_berat_pct"]],
                on="kode_bps", how="inner", validate="one_to_one")
         .merge(san_j[["kode_bps", "sanitasi"]],
                on="kode_bps", how="inner", validate="one_to_one")
         .merge(kep_j[["kode_bps", "kepadatan"]],
                on="kode_bps", how="inner", validate="one_to_one"))

assert len(final) == 38, f"harus 38 kabupaten/kota, dapat {len(final)}"
assert final["kode_bps"].is_unique, "kode_bps duplikat"
assert final.isna().sum().sum() == 0, "masih ada sel kosong"

final.to_csv(PROC / f"dataset_final_{TAHUN}.csv", index=False)
print(f"bentuk: {final.shape} | sel kosong: {int(final.isna().sum().sum())}")
print("tersimpan:", PROC / f"dataset_final_{TAHUN}.csv", "\n")

ringkas = pd.DataFrame({
    "min": final.min(numeric_only=True),
    "median": final.median(numeric_only=True),
    "max": final.max(numeric_only=True),
    "skew": final.skew(numeric_only=True),
}).round(2)
ringkas

bentuk: (38, 6) | sel kosong: 0
tersimpan: E:\DATA WRANGLING\CLustering ISPA_UAS\data\processed\dataset_final_2024.csv 



,min,median,max,skew
iku,71.91,89.00,94.84,-1.14
rate_pneu,9.97,31.30,104.46,1.82
prop_berat_pct,37.86,47.55,52.22,-1.09
sanitasi,56.95,84.11,97.43,-0.87
kepadatan,411.00,862.50,8698.00,1.75


## Bagian 5 — Clustering: K-Means vs Agglomerative Ward

Label belum ada, jadi kategori dibentuk lebih dulu lewat clustering **hanya dari Blok A**. Pra-proses mengikuti F.2.6:

1. Variabel dengan kemencengan positif > 1 ditransformasi `log1p` (meredam ekor kanan).
2. Semua variabel di-*z-score* (`StandardScaler`) agar skala per 1.000 balita dan persen setara.

Jumlah cluster diuji k = 2..6 dengan **silhouette score**; k = 3 tetap dipakai karena kategori yang diminta rendah/sedang/tinggi.

In [9]:
df = pd.read_csv(PROC / f"dataset_final_{TAHUN}.csv", dtype={"kode_bps": str})

BLOK_A = ["rate_pneu", "prop_berat_pct"]
BLOK_B = ["iku", "sanitasi", "kepadatan"]

Xa = df[BLOK_A].copy()
for c in BLOK_A:
    sk = Xa[c].skew()
    if sk > 1:
        Xa[c] = np.log1p(Xa[c])
        print(f"{c:16s} skew={sk:+.2f} -> log1p")
    else:
        print(f"{c:16s} skew={sk:+.2f} -> tetap")

Za = StandardScaler().fit_transform(Xa)


def uji_k(Z, k_range=range(2, 7)):
    baris = []
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=50, random_state=42).fit(Z)
        wd = AgglomerativeClustering(n_clusters=k, linkage="ward").fit(Z)
        baris.append({"k": k,
                      "kmeans": silhouette_score(Z, km.labels_),
                      "ward": silhouette_score(Z, wd.labels_)})
    return pd.DataFrame(baris)


tabel_k = uji_k(Za)
print("\nsilhouette score per jumlah cluster:")
tabel_k.round(3)

rate_pneu        skew=+1.82 -> log1p
prop_berat_pct   skew=-1.09 -> tetap



silhouette score per jumlah cluster:


,k,kmeans,ward
0,2,0.377,0.290
1,3,0.364,0.359
2,4,0.381,0.372
3,5,0.413,0.419
4,6,0.395,0.364


## Bagian 6 — Pelabelan rendah / sedang / tinggi

Metode dengan silhouette tertinggi pada k = 3 dipakai. Cluster diberi nama berdasarkan **rata-rata `rate_pneu`**: cluster dengan rata-rata terendah = `rendah`, tertinggi = `tinggi`. Penamaan ini hanya kosmetik — struktur cluster tidak berubah.

Stabilitas diuji dengan 50 seed K-Means berbeda, diukur **Adjusted Rand Index (ARI)** terhadap label utama: 1,0 = identik, > 0,8 = sangat stabil.

In [10]:
skor3 = tabel_k[tabel_k.k == 3].iloc[0]
metode = "kmeans" if skor3.kmeans >= skor3.ward else "ward"
print(f"k=3 -> KMeans={skor3.kmeans:.3f} | Ward={skor3.ward:.3f}")
print(f"metode terpilih: {metode.upper()}\n")

if metode == "kmeans":
    lab = KMeans(n_clusters=3, n_init=50, random_state=42).fit_predict(Za)
else:
    lab = AgglomerativeClustering(n_clusters=3, linkage="ward").fit_predict(Za)


def label_urut(df, labels):
    s = pd.Series(labels, index=df.index)
    urut = df.groupby(s)["rate_pneu"].mean().sort_values().index.tolist()
    peta = {c: n for c, n in zip(urut, ["rendah", "sedang", "tinggi"])}
    return s.map(peta)


df["label"] = label_urut(df, lab)

profil = df.groupby("label").agg(
    n=("kode_bps", "size"),
    rate_pneu=("rate_pneu", "mean"),
    prop_berat=("prop_berat_pct", "mean"),
    iku=("iku", "mean"),
    sanitasi=("sanitasi", "mean"),
    kepadatan=("kepadatan", "mean"),
).reindex(["rendah", "sedang", "tinggi"])
profil.round(2)

k=3 -> KMeans=0.364 | Ward=0.359
metode terpilih: KMEANS



,n,rate_pneu,prop_berat,iku,sanitasi,kepadatan
label,,,,,,
rendah,13,22.57,47.28,89.24,78.45,761.69
sedang,16,40.74,49.40,86.40,84.87,1890.31
tinggi,9,49.05,43.50,89.42,84.80,3790.33


In [11]:
def stabilitas(Z, k, acuan, n=50):
    skor = [adjusted_rand_score(
        acuan, KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(Z))
        for s in range(n)]
    return float(np.mean(skor)), float(np.min(skor))


rata, minimum = stabilitas(Za, 3, lab)
print(f"ARI rata-rata 50 seed = {rata:.3f} | ARI terendah = {minimum:.3f}")
print("(> 0,8 = struktur cluster sangat stabil terhadap perubahan seed)")

ARI rata-rata 50 seed = 0.926 | ARI terendah = 0.679
(> 0,8 = struktur cluster sangat stabil terhadap perubahan seed)


## Bagian 7 — Klasifikasi pembanding

Tugas klasifikasi: memprediksi label hasil clustering **hanya dari Blok B** (`iku`, `sanitasi`, `kepadatan`) — variabel yang bisa diamati tanpa menghitung outcome.

Karena n = 38, validasi memakai **Leave-One-Out CV** (setiap kab/kota jadi data uji sekali). Metrik: akurasi dan macro-F1 (macro-F1 memperlakukan ketiga kelas setara, penting karena jumlah anggota cluster tidak seimbang). Empat model dibandingkan: Random Forest, Regresi Logistik, KNN-5, dan Decision Tree.

Variabel Blok B dengan kemencengan positif lebih dari 1 (`kepadatan`, +1,75) ditransformasi $\log(1+x)$ lebih dulu, mengikuti aturan yang sama pada F.2.6; `iku` dan `sanitasi` menceng ke kiri sehingga dibiarkan pada skala aslinya.

Hyperparameter dikunci di awal (bukan hasil tuning pada data ini) agar skor tidak optimistis.

In [12]:
# aturan yang sama seperti F.2.6: log1p untuk kemencengan positif > 1
Xb = df[BLOK_B].copy()
for c in BLOK_B:
    sk = Xb[c].skew()
    if sk > 1:
        Xb[c] = np.log1p(Xb[c])
        print(f"{c:10s} skew={sk:+.2f} -> log1p")
    else:
        print(f"{c:10s} skew={sk:+.2f} -> tetap")
Xb = Xb.to_numpy(float)

y = df["label"].to_numpy(object)
loo = LeaveOneOut()

model = {
    "Random Forest": RandomForestClassifier(n_estimators=500, max_depth=3, random_state=42),
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "KNN-5": make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "Tree-d3": DecisionTreeClassifier(max_depth=3, random_state=42),
}

hasil = []
for nama, m in model.items():
    pred = cross_val_predict(m, Xb, y, cv=loo)
    hasil.append({"model": nama,
                  "akurasi": accuracy_score(y, pred),
                  "macroF1": f1_score(y, pred, average="macro")})

tabel_model = pd.DataFrame(hasil).sort_values("macroF1", ascending=False)
print(f"baseline tebak acak 3 kelas = {1/3:.3f}\n")
tabel_model.round(3)

iku        skew=-1.14 -> tetap
sanitasi   skew=-0.87 -> tetap
kepadatan  skew=+1.75 -> log1p


baseline tebak acak 3 kelas = 0.333



,model,akurasi,macroF1
1,LogReg,0.553,0.555
3,Tree-d3,0.500,0.504
2,KNN-5,0.447,0.473
0,Random Forest,0.342,0.364


In [13]:
rf = RandomForestClassifier(n_estimators=500, max_depth=3, random_state=42).fit(Xb, y)

print("feature importance (RF, dilatih pada seluruh data):")
for c, v in sorted(zip(BLOK_B, rf.feature_importances_), key=lambda t: -t[1]):
    print(f"  {c:10s} {v:.3f}")

pred_rf = cross_val_predict(rf, Xb, y, cv=loo)
print("\nlaporan per kelas (RF, LOOCV):")
print(classification_report(y, pred_rf, zero_division=0, digits=3))

feature importance (RF, dilatih pada seluruh data):
  kepadatan  0.352
  iku        0.325
  sanitasi   0.323



laporan per kelas (RF, LOOCV):
              precision    recall  f1-score   support

      rendah      0.357     0.385     0.370        13
      sedang      0.250     0.250     0.250        16
      tinggi      0.500     0.444     0.471         9

    accuracy                          0.342        38
   macro avg      0.369     0.360     0.364        38
weighted avg      0.346     0.342     0.343        38



## Bagian 8 — Simpan keluaran

Tabel akhir berisi 38 kabupaten/kota lengkap dengan `label` dan kelima variabel, untuk dipakai di bab hasil dan lampiran.

In [14]:
keluaran = dim.merge(df[["kode_bps", "label"] + BLOK_A + BLOK_B],
                    on="kode_bps", how="left")
keluaran.to_csv(OUT / f"cluster_labels_{TAHUN}.csv", index=False)

print(f"bentuk: {keluaran.shape} | label kosong: {keluaran['label'].isna().sum()}")
print("distribusi label:")
print(keluaran["label"].value_counts().reindex(["rendah", "sedang", "tinggi"]).to_string())
print("\ntersimpan:", OUT / f"cluster_labels_{TAHUN}.csv")

keluaran.sort_values(["label", "rate_pneu"]).reset_index(drop=True)

bentuk: (38, 11) | label kosong: 0
distribusi label:
label
rendah    13
sedang    16
tinggi     9

tersimpan: E:\DATA WRANGLING\CLustering ISPA_UAS\output\cluster_labels_2024.csv


,kode_bps,nama_wilayah,jenis,provinsi,tahun,label,rate_pneu,prop_berat_pct,iku,sanitasi,kepadatan
0,3527,Sampang,Kabupaten,Jawa Timur,2024,rendah,9.97,47.43,91.18,89.50,827
1,3501,Pacitan,Kabupaten,Jawa Timur,2024,rendah,12.46,46.64,92.06,81.97,411
2,3506,Kediri,Kabupaten,Jawa Timur,2024,rendah,18.19,47.82,94.31,82.39,1109
3,3513,Probolinggo,Kabupaten,Jawa Timur,2024,rendah,20.34,47.30,88.71,63.79,687
4,3508,Lumajang,Kabupaten,Jawa Timur,2024,rendah,20.95,48.20,92.07,79.22,638
5,3509,Jember,Kabupaten,Jawa Timur,2024,rendah,21.15,48.95,92.37,68.39,786
6,3510,Banyuwangi,Kabupaten,Jawa Timur,2024,rendah,21.51,47.18,87.02,89.89,488
7,3529,Sumenep,Kabupaten,Jawa Timur,2024,rendah,23.86,49.13,90.83,69.66,553
8,3519,Madiun,Kabupaten,Jawa Timur,2024,rendah,27.70,47.46,91.61,85.11,681
9,3502,Ponorogo,Kabupaten,Jawa Timur,2024,rendah,28.32,46.26,94.09,86.18,679


## Cara membaca hasil

- **Silhouette k = 3** sekitar 0,36 — struktur cluster jelas tapi tidak rapat; wajar untuk data sosial-ekonomi 38 unit.
- **ARI ~0,93** berarti pembagian cluster hampir tidak berubah walau seed diganti: labelnya stabil, bukan kebetulan satu kali jalan.
- **Klasifikasi**: bandingkan akurasi dengan baseline 0,333. Model yang menang tipis (LogReg) menunjukkan sinyal ada tapi lemah.
- **Feature importance** dan korelasi: `kepadatan` adalah satu-satunya prediktor dengan hubungan bermakna terhadap outcome. Artinya Blok B belum cukup menjelaskan risiko ISPA — ini temuan yang harus dilaporkan apa adanya di bab pembahasan, bukan disembunyikan.

Angka di atas dihasilkan ulang setiap kali *Run All*, jadi tabel di laporan harus disalin dari output notebook ini, bukan dari ingatan.